In [5]:
# ============================================================
# Reel Match — Movie Recommender (Model + Full Website + Posters
#              + Browse All Movies grid)
# Single Jupyter cell version
# ============================================================

# pip install pandas scikit-learn flask requests   (run once, if needed)

import ast
import functools
import os
import re
import threading

import pandas as pd
import requests
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from flask import Flask, jsonify, render_template, request
from IPython.display import display, HTML

# ---------------------------------------------------------------
# CONFIG
# ---------------------------------------------------------------
PORT = 8002         # <-- changed from 5000 (was conflicting, e.g. macOS AirPlay uses 5000)
HOST = "127.0.0.1"

TMDB_API_KEY = "299532766a57dcafb2fb41d134770c9c"

POSTER_BASE_URL = "https://image.tmdb.org/t/p/w500"
PLACEHOLDER_POSTER = "https://placehold.co/500x750/16161D/9A98A6?text=No+Poster"
PAGE_SIZE = 24  # movies per page on the "Browse All" grid

# ---------------------------------------------------------------
# 1. Load & merge
# ---------------------------------------------------------------
data_movie = pd.read_csv('tmdb_5000_movies.csv')
data_credit = pd.read_csv('tmdb_5000_credits.csv')

data_credit = data_credit.rename(columns={"movie_id": "id"})
df = data_movie.merge(data_credit[["id", "cast", "crew"]], on="id")

# ---------------------------------------------------------------
# 2. Helpers to parse the stringified-JSON columns
# ---------------------------------------------------------------
def safe_literal_eval(x):
    try:
        return ast.literal_eval(x)
    except (ValueError, SyntaxError, TypeError):
        return []

def extract_names(obj_list, key="name", limit=None):
    names = [d.get(key, "") for d in obj_list if isinstance(d, dict)]
    return names[:limit] if limit else names

def extract_director(crew_list):
    for member in crew_list:
        if isinstance(member, dict) and member.get("job") == "Director":
            return [member.get("name", "")]
    return []

def clean_token(s):
    return str(s).replace(" ", "").lower()

# ---------------------------------------------------------------
# 3. Feature engineering
# ---------------------------------------------------------------
df = df.dropna(subset=["overview", "title"]).reset_index(drop=True)

for col in ["genres", "keywords", "cast", "crew"]:
    df[col] = df[col].apply(safe_literal_eval)

df["genre_names"]    = df["genres"].apply(extract_names)
df["keyword_names"]  = df["keywords"].apply(extract_names)
df["cast_names"]     = df["cast"].apply(lambda x: extract_names(x, limit=3))
df["director_name"]  = df["crew"].apply(extract_director)
df["overview_tokens"] = df["overview"].apply(lambda x: str(x).lower().split())

df["tags"] = (
    df["overview_tokens"]
    + df["genre_names"].apply(lambda lst: [clean_token(i) for i in lst])
    + df["keyword_names"].apply(lambda lst: [clean_token(i) for i in lst])
    + df["cast_names"].apply(lambda lst: [clean_token(i) for i in lst])
    + df["director_name"].apply(lambda lst: [clean_token(i) for i in lst])
)
df["tags"] = df["tags"].apply(lambda tokens: " ".join(tokens))

movies = df[[
    "id", "title", "tags", "genre_names", "overview",
    "release_date", "vote_average", "vote_count", "popularity"
]].reset_index(drop=True)

movies_by_popularity = movies.sort_values("popularity", ascending=False).reset_index(drop=True)

print(f"{len(movies)} movies ready after feature engineering")

# ---------------------------------------------------------------
# 4. Vectorize + cosine similarity
# ---------------------------------------------------------------
cv = CountVectorizer(max_features=5000, stop_words="english")
vectors = cv.fit_transform(movies["tags"]).toarray()
similarity = cosine_similarity(vectors)
print("similarity matrix shape:", similarity.shape)

# ---------------------------------------------------------------
# 5. Poster fetching (TMDB API), cached so each movie is fetched once
# ---------------------------------------------------------------
@functools.lru_cache(maxsize=None)
def get_poster_url(tmdb_id):
    if not TMDB_API_KEY or TMDB_API_KEY == "YOUR_TMDB_API_KEY_HERE":
        return PLACEHOLDER_POSTER
    try:
        resp = requests.get(
            f"https://api.themoviedb.org/3/movie/{tmdb_id}",
            params={"api_key": TMDB_API_KEY},
            timeout=5,
        )
        resp.raise_for_status()
        poster_path = resp.json().get("poster_path")
        return f"{POSTER_BASE_URL}{poster_path}" if poster_path else PLACEHOLDER_POSTER
    except Exception:
        return PLACEHOLDER_POSTER

def movie_row_to_dict(row):
    return {
        "id": int(row["id"]),
        "title": row["title"],
        "genres": row["genre_names"],
        "overview": row["overview"],
        "release_date": row["release_date"],
        "vote_average": row["vote_average"],
        "poster_url": get_poster_url(int(row["id"])),
    }

# ---------------------------------------------------------------
# 6. Recommend function (includes poster_url)
# ---------------------------------------------------------------
def recommend(movie_title, top_n=8):
    matches = movies[movies["title"].str.lower() == movie_title.lower()]
    if matches.empty:
        matches = movies[movies["title"].str.lower()
                          .str.contains(re.escape(movie_title.lower()))]
        if matches.empty:
            return []
    idx = matches.index[0]
    scores = sorted(enumerate(similarity[idx]), key=lambda x: x[1], reverse=True)
    scores = scores[1: top_n + 1]
    return [
        {**movie_row_to_dict(movies.iloc[i]), "score": round(float(s), 4)}
        for i, s in scores
    ]

for r in recommend("Avatar", top_n=5):
    print(f"  {r['title']:35s}  score={r['score']}  poster={r['poster_url'][:60]}...")

# ---------------------------------------------------------------
# 7. Write the website files (templates + static) to disk
# ---------------------------------------------------------------
os.makedirs("templates", exist_ok=True)
os.makedirs("static", exist_ok=True)

INDEX_HTML = """<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8" />
<meta name="viewport" content="width=device-width, initial-scale=1.0" />
<title>Reel Match — Movie Recommender</title>
<link rel="preconnect" href="https://fonts.googleapis.com">
<link href="https://fonts.googleapis.com/css2?family=Fraunces:opsz,wght@9..144,500;9..144,600;9..144,700&family=Work+Sans:wght@400;500;600&display=swap" rel="stylesheet">
<link rel="stylesheet" href="{{ url_for('static', filename='style.css') }}">
</head>
<body>
<div class="sprocket-strip" aria-hidden="true"></div>
<header class="hero">
  <p class="eyebrow">Content-based recommendation engine &middot; TMDB 5000</p>
  <h1>Movies Recommendation System</h1>
  <p class="subhead">Tell us a film you liked. We'll find its closest relatives by genre, cast, crew and story &mdash; no ratings required.</p>
  <div class="search-box">
    <input id="search-input" type="text" placeholder="Type a movie title... e.g. Avatar" autocomplete="off" />
    <button id="search-btn">Find matches</button>
    <button id="clear-btn" class="secondary-btn" style="display:none;">Back to all movies</button>
    <div id="suggestions" class="suggestions"></div>
  </div>
</header>
<main>
  <div id="status" class="status"></div>
  <section id="results" class="results"></section>
  <div id="pagination" class="pagination"></div>
</main>
<footer>
  <p>Similarity computed with cosine distance over genres, keywords, top cast, director and overview text. Posters via TMDB.</p>
</footer>
<div class="sprocket-strip sprocket-strip--bottom" aria-hidden="true"></div>
<script src="{{ url_for('static', filename='app.js') }}"></script>
</body>
</html>
"""

STYLE_CSS = """
:root{--bg:#0B0B0F;--surface:#16161D;--surface-2:#1D1D26;--gold:#E8B33D;--crimson:#B3423A;--text:#F5F4F0;--text-dim:#9A98A6;--border:#2A2A34;}
*{box-sizing:border-box;}
body{margin:0;background:var(--bg);color:var(--text);font-family:'Work Sans',sans-serif;min-height:100vh;}
.sprocket-strip{height:22px;background:repeating-linear-gradient(90deg,var(--surface) 0 14px,transparent 14px 34px);background-color:#000;position:relative;}
.sprocket-strip::before{content:"";position:absolute;inset:0;background-image:radial-gradient(circle 5px at 17px 11px, var(--bg) 5px, transparent 5.5px);background-repeat:repeat-x;background-size:34px 22px;}
.sprocket-strip--bottom{margin-top:48px;}
.hero{max-width:760px;margin:0 auto;padding:64px 24px 40px;text-align:center;}
.eyebrow{text-transform:uppercase;letter-spacing:.14em;font-size:.72rem;color:var(--gold);font-weight:600;margin:0 0 18px;}
.hero h1{font-family:'Fraunces',serif;font-weight:700;font-size:clamp(2.6rem,7vw,4.2rem);margin:0 0 18px;letter-spacing:-.01em;background:linear-gradient(180deg,#fff,#cfc9b8);-webkit-background-clip:text;background-clip:text;color:transparent;}
.subhead{color:var(--text-dim);font-size:1.05rem;line-height:1.6;max-width:520px;margin:0 auto 36px;}
.search-box{position:relative;display:flex;gap:10px;max-width:640px;margin:0 auto;flex-wrap:wrap;justify-content:center;}
#search-input{flex:1;min-width:220px;padding:16px 20px;border-radius:8px;border:1px solid var(--border);background:var(--surface);color:var(--text);font-size:1rem;font-family:inherit;outline:none;transition:border-color .15s;}
#search-input:focus{border-color:var(--gold);}
#search-input::placeholder{color:#6b6975;}
#search-btn{padding:16px 22px;border-radius:8px;border:none;background:var(--gold);color:#1a1305;font-weight:600;font-size:.95rem;cursor:pointer;transition:transform .12s,background .15s;white-space:nowrap;}
#search-btn:hover{background:#f0c153;transform:translateY(-1px);}
.secondary-btn{padding:16px 20px;border-radius:8px;border:1px solid var(--border);background:transparent;color:var(--text-dim);font-size:.9rem;cursor:pointer;white-space:nowrap;}
.secondary-btn:hover{border-color:var(--gold);color:var(--gold);}
.suggestions{position:absolute;top:calc(100% + 8px);left:0;right:0;background:var(--surface-2);border:1px solid var(--border);border-radius:8px;overflow:hidden;text-align:left;z-index:10;display:none;}
.suggestions.open{display:block;}
.suggestion-item{padding:12px 18px;cursor:pointer;font-size:.92rem;color:var(--text-dim);border-bottom:1px solid var(--border);}
.suggestion-item:last-child{border-bottom:none;}
.suggestion-item:hover,.suggestion-item.active{background:var(--surface);color:var(--text);}
main{max-width:1200px;margin:0 auto;padding:8px 24px 60px;min-height:200px;}
.status{text-align:center;color:var(--text-dim);font-size:.95rem;margin:24px 0;}
.status.error{color:var(--crimson);}
.results{display:grid;grid-template-columns:repeat(auto-fill,minmax(200px,1fr));gap:20px;}
.card{background:var(--surface);border:1px solid var(--border);border-radius:10px;overflow:hidden;display:flex;flex-direction:column;transition:border-color .15s,transform .15s;}
.card:hover{border-color:var(--gold);transform:translateY(-3px);}
.poster{width:100%;aspect-ratio:2/3;object-fit:cover;background:var(--surface-2);display:block;}
.card-body{padding:14px;display:flex;flex-direction:column;gap:7px;flex:1;}
.card-top{display:flex;justify-content:space-between;align-items:flex-start;gap:8px;}
.card h3{font-family:'Fraunces',serif;font-size:1rem;margin:0;line-height:1.3;}
.match-badge{flex-shrink:0;background:var(--crimson);color:#fff;font-size:.68rem;font-weight:600;padding:4px 8px;border-radius:999px;white-space:nowrap;}
.genres{color:var(--gold);font-size:.7rem;letter-spacing:.04em;text-transform:uppercase;}
.overview{color:var(--text-dim);font-size:.8rem;line-height:1.5;display:-webkit-box;-webkit-line-clamp:3;-webkit-box-orient:vertical;overflow:hidden;}
.meta{display:flex;justify-content:space-between;font-size:.74rem;color:#6b6975;margin-top:auto;padding-top:8px;border-top:1px solid var(--border);}
.pagination{display:flex;justify-content:center;align-items:center;gap:14px;margin:32px 0 8px;color:var(--text-dim);font-size:.9rem;}
.pagination button{padding:9px 16px;border-radius:6px;border:1px solid var(--border);background:var(--surface);color:var(--text);cursor:pointer;}
.pagination button:hover:not(:disabled){border-color:var(--gold);color:var(--gold);}
.pagination button:disabled{opacity:.4;cursor:default;}
footer{text-align:center;color:#54525d;font-size:.78rem;padding:0 24px 40px;}
@media (max-width:520px){.search-box{flex-direction:column;}#search-btn,#clear-btn{width:100%;}}
"""

APP_JS = """
const input = document.getElementById('search-input');
const btn = document.getElementById('search-btn');
const clearBtn = document.getElementById('clear-btn');
const suggestionsBox = document.getElementById('suggestions');
const resultsEl = document.getElementById('results');
const statusEl = document.getElementById('status');
const paginationEl = document.getElementById('pagination');

let debounceTimer = null, activeIndex = -1;
let currentPage = 1;
let mode = 'browse';

window.addEventListener('DOMContentLoaded', () => loadBrowsePage(1));

async function loadBrowsePage(page) {
  mode = 'browse';
  currentPage = page;
  clearBtn.style.display = 'none';
  statusEl.textContent = 'Loading movies...';
  statusEl.classList.remove('error');
  try {
    const res = await fetch(`/api/movies?page=${page}`);
    const data = await res.json();
    statusEl.textContent = `Showing ${data.movies.length} of ${data.total} movies (page ${data.page} of ${data.total_pages})`;
    renderResults(data.movies, false);
    renderPagination(data.page, data.total_pages, loadBrowsePage);
  } catch (e) {
    statusEl.textContent = 'Could not reach the server.';
    statusEl.classList.add('error');
  }
}

input.addEventListener('input', () => {
  clearTimeout(debounceTimer);
  const q = input.value.trim();
  if (!q) { closeSuggestions(); return; }
  debounceTimer = setTimeout(() => fetchSuggestions(q), 200);
});

input.addEventListener('keydown', (e) => {
  const items = [...suggestionsBox.querySelectorAll('.suggestion-item')];
  if (e.key === 'ArrowDown') { e.preventDefault(); activeIndex = Math.min(activeIndex+1, items.length-1); highlight(items); }
  else if (e.key === 'ArrowUp') { e.preventDefault(); activeIndex = Math.max(activeIndex-1, 0); highlight(items); }
  else if (e.key === 'Enter') {
    if (activeIndex >= 0 && items[activeIndex]) { input.value = items[activeIndex].textContent; closeSuggestions(); }
    runSearch();
  } else if (e.key === 'Escape') { closeSuggestions(); }
});

function highlight(items){ items.forEach((el,i)=>el.classList.toggle('active', i===activeIndex)); }

async function fetchSuggestions(q) {
  try {
    const res = await fetch(`/api/titles?q=${encodeURIComponent(q)}`);
    renderSuggestions(await res.json());
  } catch (e) { closeSuggestions(); }
}

function renderSuggestions(titles) {
  activeIndex = -1;
  if (!titles.length) { closeSuggestions(); return; }
  suggestionsBox.innerHTML = titles.map(t => `<div class="suggestion-item">${escapeHtml(t)}</div>`).join('');
  suggestionsBox.classList.add('open');
  suggestionsBox.querySelectorAll('.suggestion-item').forEach(el => {
    el.addEventListener('click', () => { input.value = el.textContent; closeSuggestions(); runSearch(); });
  });
}

function closeSuggestions(){ suggestionsBox.classList.remove('open'); suggestionsBox.innerHTML=''; }
document.addEventListener('click', (e) => { if (!suggestionsBox.contains(e.target) && e.target!==input) closeSuggestions(); });

btn.addEventListener('click', runSearch);
clearBtn.addEventListener('click', () => { input.value=''; loadBrowsePage(1); });

async function runSearch() {
  const title = input.value.trim();
  if (!title) return;
  mode = 'search';
  closeSuggestions();
  resultsEl.innerHTML = '';
  paginationEl.innerHTML = '';
  clearBtn.style.display = 'inline-block';
  statusEl.textContent = `Finding movies similar to "${title}"...`;
  statusEl.classList.remove('error');
  try {
    const res = await fetch(`/api/recommend?title=${encodeURIComponent(title)}`);
    const data = await res.json();
    if (!res.ok) { statusEl.textContent = data.error || 'Something went wrong.'; statusEl.classList.add('error'); return; }
    statusEl.textContent = `${data.recommendations.length} matches for "${data.query}"`;
    renderResults(data.recommendations, true);
  } catch (e) {
    statusEl.textContent = 'Could not reach the server.';
    statusEl.classList.add('error');
  }
}

function renderResults(movies, showScore) {
  resultsEl.innerHTML = movies.map(r => `
    <article class="card">
      <img class="poster" src="${r.poster_url}" alt="${escapeHtml(r.title)} poster" loading="lazy"
           onerror="this.src='https://placehold.co/500x750/16161D/9A98A6?text=No+Poster'" />
      <div class="card-body">
        <div class="card-top">
          <h3>${escapeHtml(r.title)}</h3>
          ${showScore ? `<span class="match-badge">${Math.round(r.score*100)}%</span>` : ''}
        </div>
        <div class="genres">${(r.genres||[]).join(' &middot; ')}</div>
        <p class="overview">${escapeHtml(r.overview || 'No overview available.')}</p>
        <div class="meta"><span>${r.release_date || '-'}</span><span>&#9733; ${r.vote_average ?? '-'}</span></div>
      </div>
    </article>
  `).join('');
}

function renderPagination(page, totalPages, onGoto) {
  paginationEl.innerHTML = `
    <button ${page<=1 ? 'disabled' : ''} id="prev-page">&larr; Prev</button>
    <span>Page ${page} of ${totalPages}</span>
    <button ${page>=totalPages ? 'disabled' : ''} id="next-page">Next &rarr;</button>
  `;
  const prevBtn = document.getElementById('prev-page');
  const nextBtn = document.getElementById('next-page');
  if (prevBtn) prevBtn.addEventListener('click', () => { onGoto(page-1); window.scrollTo({top:0, behavior:'smooth'}); });
  if (nextBtn) nextBtn.addEventListener('click', () => { onGoto(page+1); window.scrollTo({top:0, behavior:'smooth'}); });
}

function escapeHtml(str){ const d=document.createElement('div'); d.textContent=str; return d.innerHTML; }
"""

with open("templates/index.html", "w") as f:
    f.write(INDEX_HTML)
with open("static/style.css", "w") as f:
    f.write(STYLE_CSS)
with open("static/app.js", "w") as f:
    f.write(APP_JS)

# ---------------------------------------------------------------
# 8. Flask app
# ---------------------------------------------------------------
flask_app = Flask(__name__)
ALL_TITLES = sorted(movies["title"].tolist())

@flask_app.route("/")
def index():
    return render_template("index.html")

@flask_app.route("/api/titles")
def api_titles():
    q = request.args.get("q", "").lower().strip()
    if not q:
        return jsonify([])
    return jsonify([t for t in ALL_TITLES if q in t.lower()][:10])

@flask_app.route("/api/movies")
def api_movies():
    page = max(1, request.args.get("page", 1, type=int))
    total = len(movies_by_popularity)
    total_pages = max(1, -(-total // PAGE_SIZE))
    page = min(page, total_pages)
    start = (page - 1) * PAGE_SIZE
    end = start + PAGE_SIZE
    page_df = movies_by_popularity.iloc[start:end]
    return jsonify({
        "page": page,
        "page_size": PAGE_SIZE,
        "total": total,
        "total_pages": total_pages,
        "movies": [movie_row_to_dict(row) for _, row in page_df.iterrows()],
    })

@flask_app.route("/api/recommend")
def api_recommend():
    title = request.args.get("title", "").strip()
    if not title:
        return jsonify({"error": "Missing 'title' query param"}), 400
    results = recommend(title, top_n=8)
    if not results:
        return jsonify({"error": f"No movie found matching '{title}'"}), 404
    return jsonify({"query": title, "recommendations": results})

# ---------------------------------------------------------------
# 9. Run Flask in a background thread
# ---------------------------------------------------------------
def run_flask():
    flask_app.run(host=HOST, port=PORT, debug=False, use_reloader=False)

flask_thread = threading.Thread(target=run_flask, daemon=True)
flask_thread.start()

url = f"http://{HOST}:{PORT}"
print(f"\nReel Match is running at {url}")
display(HTML(f'<a href="{url}" target="_blank">Open Reel Match →</a>'))

4800 movies ready after feature engineering
similarity matrix shape: (4800, 4800)
  Titan A.E.                           score=0.2572  poster=https://image.tmdb.org/t/p/w500/el2iHk3LTJWfEnwrvcRkvWY501G....
  Small Soldiers                       score=0.2546  poster=https://image.tmdb.org/t/p/w500/2nuUjSzHsoYlRvTPmLo7m7gCQry....
  Ender's Game                         score=0.2476  poster=https://image.tmdb.org/t/p/w500/pVcRI5YKnkkgaAD876jBeKb189d....
  Independence Day                     score=0.2472  poster=https://image.tmdb.org/t/p/w500/p0BPQGSPoSa8Ml0DAf2mB2kCU0R....
  Aliens vs Predator: Requiem          score=0.2446  poster=https://image.tmdb.org/t/p/w500/5iTwPDNtvK6ZZF607BHBbU3HO0B....

Reel Match is running at http://127.0.0.1:8002


 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on http://127.0.0.1:8002
Press CTRL+C to quit
127.0.0.1 - - [14/Aug/2026 13:20:14] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [14/Aug/2026 13:20:15] "GET /static/style.css HTTP/1.1" 200 -
127.0.0.1 - - [14/Aug/2026 13:20:15] "GET /static/app.js HTTP/1.1" 200 -
127.0.0.1 - - [14/Aug/2026 13:20:15] "GET /favicon.ico HTTP/1.1" 404 -
127.0.0.1 - - [14/Aug/2026 13:20:22] "GET /api/titles?q=d HTTP/1.1" 200 -
127.0.0.1 - - [14/Aug/2026 13:20:22] "GET /api/titles?q=dh HTTP/1.1" 200 -
127.0.0.1 - - [14/Aug/2026 13:20:29] "GET /api/recommend?title=Gandhi,%20My%20Father HTTP/1.1" 200 -
127.0.0.1 - - [14/Aug/2026 13:20:31] "GET /api/movies?page=1 HTTP/1.1" 200 -
127.0.0.1 - - [14/Aug/2026 13:21:07] "GET /api/movies?page=2 HTTP/1.1" 200 -
